<a href="https://colab.research.google.com/github/shivaxdynamic-hash/telecom-churn/blob/main/telecomchurn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
#load dataset
df=pd.read_csv("/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())
print(df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [ ]:
#check null values
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [ ]:
#no neeed to use dropna and fullna
#now checking duplicates
print(df.duplicated().sum())
print("shape of dataset:",df.shape)

0
shape of dataset: (7043, 21)


In [ ]:
#finding outliers
#IQR method
num_cols=df.select_dtypes(include=np.number).columns
for col in num_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_limit=Q1-1.5*IQR
  upper_limit=Q3+1.5*IQR
  outliers = df[(df[col]<lower_limit)|(df[col]>upper_limit)]
  print(f"{col}:{len(outliers)} rows detected as outliers")

SeniorCitizen:1142 rows detected as outliers
tenure:0 rows detected as outliers
MonthlyCharges:0 rows detected as outliers


In [ ]:
#now we have to remove outliers from senior citizen column
num_cols=df.select_dtypes(include=np.number).columns
for col in num_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_limit=Q1-1.5*IQR
  upper_limit=Q3+1.5*IQR
  df=df[(df[col]>=lower_limit)&(df[col]<=upper_limit)]
print("shape after removing outliers:",df.shape)

shape after removing outliers: (5901, 21)


In [ ]:
le=LabelEncoder()
cat_cols=df.select_dtypes(include='object').columns
for col in cat_cols:
  df[col]=le.fit_transform(df[col])
print(df.head())

   customerID  gender  SeniorCitizen  Partner  Dependents  tenure  \
0        4490       0              0        1           0       1   
1        3306       1              0        0           0      34   
2        2155       1              0        0           0       2   
3        4631       1              0        0           0      45   
4        5445       0              0        0           0       2   

   PhoneService  MultipleLines  InternetService  OnlineSecurity  ...  \
0             0              1                0               0  ...   
1             1              0                0               2  ...   
2             1              0                0               2  ...   
3             0              1                0               2  ...   
4             1              0                1               0  ...   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0                 0            0            0                0         0   


In [ ]:
#input and output split
X=df.drop("Churn",axis=1)#feature
Y=df["Churn"]
print("input columns:",X.head())


input columns:    customerID  gender  SeniorCitizen  Partner  Dependents  tenure  \
0        4490       0              0        1           0       1   
1        3306       1              0        0           0      34   
2        2155       1              0        0           0       2   
3        4631       1              0        0           0      45   
4        5445       0              0        0           0       2   

   PhoneService  MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0             0              1                0               0             2   
1             1              0                0               2             0   
2             1              0                0               2             2   
3             0              1                0               2             0   
4             1              0                1               0             0   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0      

In [ ]:
#train and test
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(
    X,Y,test_size=0.2,random_state=42)

In [ ]:
#featurescaling
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)
print(X_train)

[[-1.46990325  0.98402587  0.         ...  1.28638464  0.16308548
  -1.63421565]
 [ 0.21972315 -1.01623344  0.         ...  1.28638464  1.11535789
  -1.59005807]
 [ 0.75276355  0.98402587  0.         ...  1.28638464  0.28377278
  -0.51702877]
 ...
 [ 0.9890174   0.98402587  0.         ...  0.37306703  0.42099259
  -0.74160163]
 [ 1.42861411 -1.01623344  0.         ...  0.37306703  1.49229905
   0.00592318]
 [-1.2806651   0.98402587  0.         ... -1.45356821  0.32510405
  -0.86398122]]


In [ ]:
#model selction
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC()
}

for name, model in models.items():
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    acc = (Y_pred == Y_test).mean()#find fraction of all true values
    print(f"{name}: {acc:.4f}")

Logistic Regression: 0.8146
Decision Tree: 0.7282
Random Forest: 0.8078
KNN: 0.7705
SVM: 0.8239


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score,recall_score,f1_score

cm = confusion_matrix(Y_test, Y_pred)
print(cm)
print(classification_report(Y_test, Y_pred))  # precision, recall, f1, support
print(accuracy_score(Y_test, Y_pred))  # accuracy

[[864  52]
 [156 109]]
              precision    recall  f1-score   support

           0       0.85      0.94      0.89       916
           1       0.68      0.41      0.51       265

    accuracy                           0.82      1181
   macro avg       0.76      0.68      0.70      1181
weighted avg       0.81      0.82      0.81      1181

0.8238780694326842
